In [ ]:
import os 
import numpy as np
from pathlib import Path
from scipy.signal import resample
import scipy.stats as stats
from scipy.stats import binned_statistic
import matplotlib.pyplot as plt

import importlib
import re
import subprocess

import cellTV_functions as cellTV
import parse_session_functions
import neural_analysis_helpers
from roicat_analysis import roi_tracking_helpers

importlib.reload(roi_tracking_helpers)
importlib.reload(parse_session_functions)
importlib.reload(neural_analysis_helpers)
importlib.reload(cellTV)


In [ ]:
# Define paths
mouse = 'TAA0000066'
base_path = Path('/Volumes/mrsic_flogel/public/projects/AtApSuKuSaRe_20250129_HFScohort2/')

# Path to roicat environment's python (roicat runs on python >= 3.13)
roicat_env_python = "/Users/athinaapostolelli/miniforge3/envs/roicat/bin/python"

tracked_neurons = {}

#### Load roicat tracking and alignment data 

In [ ]:
# Load roicat tracked neurons
roicat_data_name = mouse 
roicat_dir = base_path / f'{roicat_data_name}' / 'ROICaT'

# Choose sessions and paths for alignment
sessions_to_align = ['ses-005_date-20250218_protocol-t0',
    'ses-006_date-20250224_protocol-t1',
    'ses-007_date-20250304_protocol-t2',
    'ses-008_date-20250306_protocol-t3',
    'ses-010_date-20250314_protocol-t4',
    'ses-011_date-20250315_protocol-t5',
    'ses-012_date-20250318_protocol-t6',
    'ses-013_date-20250320_protocol-t7',
    'ses-014_date-20250326_protocol-t8',
    'ses-015_date-20250327_protocol-t9',
    'ses-016_date-20250330_protocol-t10',
    'ses-017_date-20250331_protocol-t11',
    'ses-018_date-20250403_protocol-t12',
    'ses-019_date-20250404_protocol-t13',
    'ses-020_date-20250412_protocol-t14',
    'ses-021_date-20250426_protocol-t15',
    'ses-022_date-20250509_protocol-t16',
    'ses-023_date-20250516_protocol-t17']

protocols = [int(re.search(r'protocol-t(\d+)', s).group(1)) for s in sessions_to_align]
protocols = [protocols[0], protocols[-1]]

alignment_dir = roicat_dir / 'alignment' / '_'.join(f't{n}' for n in protocols)
filename = f"roicat_aligned_ROIs_{'_'.join(['t' + str(n) for n in protocols])}.npy"

# Load aligned cluster ids
_, idx_original_aligned = roi_tracking_helpers.align_rois(roicat_dir, roicat_data_name, alignment_dir=alignment_dir, filename=filename)

In [ ]:
def get_goal_progress_cells(dF, neurons, session, event_frames, stage, save_path, ngoals=4, bins=90, reload=False, plot=True, shuffle=False):
    # Find goal progress tuned cells - takes long if shuffling
    filename = f'T{stage}_{ngoals}goal_progress_tracked_neurons.npz'

    if os.path.exists(os.path.join(save_path, filename)) and not reload:
        print(f'Goal progress and tracked neurons found. Loading...')
        goal_progress_tuned = np.load(os.path.join(save_path, filename))['goal_progress_tuned']

    else:
        goal_progress_tuned = []
        for cell in neurons:
            real_score, shuffled_scores, phase_pref, state_pref = cellTV.calc_goal_tuningix(dF, cell, session, condition='arb', event_frames=event_frames, n_goals=ngoals, frame_rate=45, bins=bins, shuffle=shuffle, plot=False)

            if (real_score > 1) & (np.abs(real_score - np.median(shuffled_scores)) > 0.5):
                goal_progress_tuned.append(cell)

        # Plot firing rates for goal progress tuned cells
        for cell in goal_progress_tuned:
            _ = cellTV.extract_arb_progress(dF, cell, event_frames, ngoals=ngoals, bins=90, plot=plot, shuffle=False)

        # Save these neurons
        np.savez(os.path.join(save_path, filename), goal_progress_tuned=np.array(goal_progress_tuned))

    print(f"{len(goal_progress_tuned)} out of {len(neurons)} tracked neurons are goal progress tuned in T{stage}")

    return goal_progress_tuned

#### T8

In [ ]:
# Define paths
stage = '-t8'

session_folder = [f for f in os.listdir(os.path.join(base_path, mouse)) if stage in f][0]
sess_data_path = os.path.join(base_path, mouse, session_folder)

imaging_path, config_path, frame_ix, date1, date2 = cellTV.get_session_folders(base_path, mouse, stage)

save_path = Path(sess_data_path) / 'analysis'
save_path.mkdir(parents=True, exist_ok=True)

reload_goal_progress = False

In [ ]:
# Load dF
calculate_DF_F = False

DF_F_file = os.path.join(imaging_path, 'DF_F0.npy')

if os.path.exists(DF_F_file) and calculate_DF_F is False:
    print('DF_F0 file found. Loading...')
    
    DF_F_all = np.load(DF_F_file)
    DF_F = DF_F_all[:, frame_ix]
    print(DF_F.shape)
    
else:
    # TODO: incorporate this into the main analysis and ensure DF_F is the same everywhere
    f, fneu, iscell, ops, seg, frame_rate = cellTV.load_img_data(imaging_path)
    dF_8 = cellTV.get_dff(f, fneu, frame_ix, ops)
    

In [ ]:
# Get session data 
if stage in ['-t3','-t4','-t5', '-t6']:
    session8 = parse_session_functions.analyse_npz_pre7(mouse, date2, plot=False)
else:
    session8 = parse_session_functions.analyse_npz(mouse, date2, plot=True)

In [ ]:
# Goal progress 

# Get the neurons in this session that have been tracked with roicat
session_idx = [s for s in range(len(idx_original_aligned)) if str(s) in stage][0] # session idx in the aligned neuron array

tracked_neurons[session_idx] = idx_original_aligned[session_idx][~np.isnan(idx_original_aligned[session_idx])].astype(int)

# Find goal progress tuned cells - takes long if shuffling
goal_progress_tuned_t8 = get_goal_progress_cells(dF_8, tracked_neurons[session_idx], session8, \
                                              event_frames=session8['reward_idx'], stage=session_idx, \
                                                save_path=save_path, ngoals=4, bins=90, plot=True, \
                                                    shuffle=True, reload=reload_goal_progress)

#### T6

In [ ]:
# Define paths
stage = '-t6'

session_folder = [f for f in os.listdir(os.path.join(base_path, mouse)) if stage in f][0]
sess_data_path = os.path.join(base_path, mouse, session_folder)

imaging_path, config_path, frame_ix, date1, date2 = cellTV.get_session_folders(base_path, mouse, stage)

save_path6 = Path(sess_data_path) / 'analysis'
save_path6.mkdir(parents=True, exist_ok=True)

reload_goal_progress = False

In [ ]:
# Load dF
calculate_DF_F = False

DF_F_file = os.path.join(imaging_path, 'DF_F0.npy')

if os.path.exists(DF_F_file) and calculate_DF_F is False:
    print('DF_F0 file found. Loading...')
    
    DF_F_all = np.load(DF_F_file)
    dF_6 = DF_F_all[:, frame_ix['valid_frames']]
    print(dF_6.shape)
    
else:
    # TODO: incorporate this into the main analysis and ensure DF_F is the same everywhere
    f, fneu, iscell, ops, seg, frame_rate = cellTV.load_img_data(imaging_path)
    dF_6 = cellTV.get_dff(f, fneu, frame_ix, ops)
    

In [ ]:
# Get session data 
if stage in ['-t3','-t4','-t5', '-t6']:
    session6 = parse_session_functions.analyse_npz_pre7(mouse, date2, stage='t6', plot=False)
else:
    session6 = parse_session_functions.analyse_npz(mouse, date2, plot=True)

# Get lick profile 
neural_analysis_helpers.plot_lick_maps(session6)

# Get speed profile
parse_session_functions.plot_speed_profile(session6, stage=6)

# Get speed and lick rate as fake neurons and plot goal progress
lick_neuron = np.array(session6['frame_lick_rate']).reshape(1, -1)
speed_neuron = np.array(session6['speed']).reshape(1, -1)

event_idx = np.sort(np.concatenate([session6['reward_idx'], session6['miss_rew_idx'], session6['test_rew_idx']])).astype(int)

_ = cellTV.plot_arb_progress_2cells(dF=[lick_neuron, speed_neuron], cell=[0, 0], 
                                event_frames=[event_idx, event_idx], ngoals=5, 
                                bins=90, stages=np.array([6, 6]), labels=['T6 - lick rate', 'T6 - speed'], 
                                plot=True, shuffle=False)

In [ ]:
# Goal progress - 4 goals

# Get the neurons in this session that have been tracked with roicat
session_idx = [s for s in range(len(idx_original_aligned)) if str(s) in stage][0] # session idx in the aligned neuron array

tracked_neurons[session_idx] = idx_original_aligned[session_idx][~np.isnan(idx_original_aligned[session_idx])].astype(int)

# Find goal progress tuned cells 
goal_progress_tuned_t6_4g = get_goal_progress_cells(dF_6, tracked_neurons[session_idx], session6, \
                                              event_frames=session6['reward_idx'], stage=session_idx, \
                                                save_path=save_path6, ngoals=4, bins=90, plot=True, \
                                                    shuffle=True, reload=reload_goal_progress)

In [ ]:
# Goal progress - 5 goals

# Get the neurons in this session that have been tracked with roicat
session_idx = [s for s in range(len(idx_original_aligned)) if str(s) in stage][0] # session idx in the aligned neuron array

tracked_neurons[session_idx] = idx_original_aligned[session_idx][~np.isnan(idx_original_aligned[session_idx])].astype(int)

# Find goal progress tuned cells 
event_idx = np.sort(np.concatenate([session6['reward_idx'], session6['miss_rew_idx'], session6['test_rew_idx']])).astype(int)

goal_progress_tuned_t6_5g = get_goal_progress_cells(dF_6, tracked_neurons[session_idx], session6, \
                                              event_frames=event_idx, stage=session_idx, \
                                                save_path=save_path6, ngoals=5, bins=90, plot=True, \
                                                    shuffle=True, reload=reload_goal_progress)

##### Suppression signal analysis

1. Find cells with a single strong test peak

In [ ]:
# Define paths and parms
stage = 6
bins = 30
nreps = 100
shuffle = False
plot_mean_vs_shuffled = False
reload_goal_tuned = False 
reload_goal_lmEntry_tuned = False
goal_by_lmEntry = False

# Select data
session = session6
dF = dF_6
session_idx = stage # session idx in the aligned neuron array
save_path = save_path6

# Define saving paths
test_lm_path = Path(save_path) / 'test_lm_cells' 
test_lm_path.mkdir(parents=True, exist_ok=True)

goal_tuned_path = os.path.join(test_lm_path, f"t{session_idx}_test_goal_tuned.npz")
lm_firing_path = os.path.join(test_lm_path, f't{session_idx}_lm_firing.npz')
goal_lmEntry_firing_path = os.path.join(test_lm_path, f't{session_idx}_goal_lmExit_firing.npz')
goal_firing_path = os.path.join(test_lm_path, f't{session_idx}_goal_firing.npz')
before_lm_firing_path = os.path.join(test_lm_path, f"t{session_idx}_before_lm_firing.npz")

# Load cell IDs if already calculated 
if os.path.exists(goal_tuned_path) and not reload_goal_tuned:
    print(f'Found test-tuned cells for T{session_idx}. Loading ...')
    data = np.load(goal_tuned_path, allow_pickle=True)
    test_peak_cells_t6 = data["high_test_goal_cells"]
else:
    print(f'Did not find test-tuned cells for T{session_idx}.')
    reload_goal_tuned = True


In [ ]:
# Bin neural activity into 5 goals using landmark occupancy indices
# goal_lms = [1,3,5,7,9]
goal_lms = [0,2,4,6,8]

ngoals = len(goal_lms)
bins = 90
nbins = ngoals * bins

# Choose binning data and paths
if goal_by_lmEntry: 
    # Non-goal landmark entry indices are used to bin activity so that 
    print('Using landmark entry events for goal binning')

    lm_entry_idx, lm_exit_idx = neural_analysis_helpers.get_lm_entry_exit(session)
    # event_idx = np.sort(np.concatenate([lm_exit_idx[i::session['num_landmarks']] for i in goal_lms]))
    event_idx = np.sort(np.concatenate([lm_entry_idx[i::session['num_landmarks']] for i in goal_lms]))

    binned_firing_path = goal_lmEntry_firing_path
    reload_flag = reload_goal_lmEntry_tuned
    
else:
    # Reward indices are used to bin activity
    print('Using reward events for goal binning')
    
    event_idx = np.sort(np.concatenate([session['reward_idx'], session['miss_rew_idx'], session['test_rew_idx']])).astype(int)

    binned_firing_path = goal_firing_path
    reload_flag = reload_goal_tuned

# Bin data
nlaps = len(event_idx) // ngoals

tracked_neurons[session_idx] = idx_original_aligned[session_idx][~np.isnan(idx_original_aligned[session_idx])].astype(int)

if os.path.exists(binned_firing_path) and not reload_flag:
    print(f'Found goal activity for T{session_idx}. Loading ...')
    data = np.load(binned_firing_path, allow_pickle=True)
    mean_goal_firing = data["mean_goal_firing"].item()
    shuffled_goal_firing = data["shuffled_goal_firing"].item()

else:
    mean_goal_firing = {}
    shuffled_goal_firing = {}

    for cell in tracked_neurons[session_idx]: 
        if shuffle:
            mean_goal_firing[cell] = cellTV.extract_arb_progress(dF, cell, event_idx, ngoals=ngoals, bins=bins, stage=stage, plot=False, shuffle=False)
            
            shuffled_goal_firing[cell] = np.empty((nreps, nlaps, nbins))
            for i in range(nreps):
                shuffled_goal_firing[cell][i] = cellTV.extract_arb_progress(dF, cell, event_idx, ngoals=ngoals, bins=bins, stage=stage, plot=False, shuffle=shuffle)
        else:
            mean_goal_firing[cell] = cellTV.extract_arb_progress(dF, cell, event_idx, ngoals=ngoals, bins=bins, stage=stage, plot=False, shuffle=False)

    # Save data 
    np.savez(binned_firing_path, 
            mean_goal_firing=mean_goal_firing, 
            shuffled_goal_firing=shuffled_goal_firing)
    
mean_goal_firing6 = mean_goal_firing

In [ ]:
# Identify the cells with a single strong test peak
neurons = tracked_neurons[session_idx]
test_peak_cells_t6 = neural_analysis_helpers.get_test_peak_tuned_cells(dF, mean_goal_firing6, event_idx, 
                                                                           session_idx, neurons, 
                                                                           bins=90, rew_goals=[0,1,2,4], 
                                                                           test_goal=3, session=session,
                                                                           save_path=goal_tuned_path, plot=True, 
                                                                           add_lick_rate=True)
# TODO savefig 
# plt.tight_layout()
# plt.savefig(os.path.join(save_path, f'T5split_T6_cell{c}_{cell5}_{cell6}.png'), transparent=True, dpi=300)

# Find fraction of these cells across all session tracked cells 
perc_test_tuned_t6 = len(test_peak_cells_t6) / len(neurons)
print(perc_test_tuned_t6)

2. Find cells with 5 peaks but with a strong test peak

#### T5

In [ ]:
# Define paths
stage = '-t5'

session_folder = [f for f in os.listdir(os.path.join(base_path, mouse)) if stage in f][0]
sess_data_path = os.path.join(base_path, mouse, session_folder)

imaging_path, config_path, frame_ix, date1, date2 = cellTV.get_session_folders(base_path, mouse, stage)

save_path5 = Path(sess_data_path) / 'analysis'
save_path5.mkdir(parents=True, exist_ok=True)

reload_goal_progress = False

In [ ]:
# Load dF
calculate_DF_F = False

DF_F_file = os.path.join(imaging_path, 'DF_F0.npy')

if os.path.exists(DF_F_file) and calculate_DF_F is False:
    print('DF_F0 file found. Loading...')
    
    DF_F_all = np.load(DF_F_file)
    dF_5 = DF_F_all[:, frame_ix['valid_frames']]
    print(dF_5.shape)
    
else:
    # TODO: incorporate this into the main analysis and ensure DF_F is the same everywhere
    f, fneu, iscell, ops, seg, frame_rate = cellTV.load_img_data(imaging_path)
    dF_5 = cellTV.get_dff(f, fneu, frame_ix, ops)
    

In [ ]:
# Get session data TODO: BIG FIX

if stage in ['-t3','-t4','-t5', '-t6']:
    session5 = parse_session_functions.analyse_npz_pre7(mouse, date2, stage='t5', plot=False)
else:
    session5 = parse_session_functions.analyse_npz(mouse, date2, plot=True)
# if stage in ['-t3','-t4','-t5', '-t6']:
#     session = parse_session_functions.analyse_npz_pre7(mouse, date2, stage, plot=False)
#     if mouse == 'TAA0000066' or mouse == 'TAA0000059':
#         if int(stage[-1]) == 3 or int(stage[-1]) == 4:
#             sequence = 'AB_shuffled'
#         elif int(stage[-1]) == 5 or int(stage[-1]) == 6:
#             sequence = 'ABAB'
#         else:
#             print('This code does not work before T3 or beyond T6 yet.')
#     elif mouse == 'TAA0000061' or mouse == 'TAA0000064':
#         if int(stage[-1]) == 3 or int(stage[-1]) == 4:
#             sequence = 'AABB'
#         elif int(stage[-1]) == 5 or int(stage[-1]) == 6:
#             sequence = 'AABB'
#         else:
#             print('This code does not work before T3 or beyond T6 yet.')
#     elif mouse == 'TAA0000062' or mouse == 'TAA0000065':
#         if int(stage[-1]) == 3 or int(stage[-1]) == 4:
#             sequence = 'AB_shuffled'
#         elif int(stage[-1]) == 5 or int(stage[-1]) == 6:
#             sequence = 'AABB'
#         else:
#             print('This code does not work before T3 or beyond T6 yet.')
#     else:
#         raise ValueError("Oops I don't know what to do about this mouse")
    # nidaq_data = neural_analysis_helpers.load_nidaq_behaviour_data(sess_data_path)

    # # Load VR data 
    # VR_data, options = neural_analysis_helpers.load_vr_behaviour_data(sess_data_path)
    # session5 = parse_session_functions.create_session_struct(VR_data, options)
    # session5 = parse_session_functions.get_num_landmarks(session5, options)
    # session5 = parse_session_functions.get_lap_idx(session5)
    # session5 = parse_session_functions.get_lm_idx(session5)
    # session5 = parse_session_functions.get_rewarded_lms(session5)
    # session5 = parse_session_functions.get_active_goal(session5)
    # session5 = parse_session_functions.calc_laps_needed(session5)
    # session5 = parse_session_functions.get_lms_visited(options, session5)
    # session5 = parse_session_functions.get_num_landmarks(session5, options)
    
    # print('Number of laps = ', session5['num_laps'])
    # num_lms = len(session5['all_landmarks'])

    # lm_entry_idx, lm_exit_idx = neural_analysis_helpers.get_lm_entry_exit(session5, positions=positions)

    # # 2. Define which landmarks belong to goals, non-goals and test
    # session5 = neural_analysis_helpers.get_landmark_categories(sequence, session5)

    # # 3. Find licks TODO might need to use get_licks function
    # # lick_idx = np.where(nidaq_data['licks'] == 1)[0]
    # session5 = neural_analysis_helpers.get_licks(nidaq_data, session5)

    # # 4. Find landmarks that were rewarded
    # session5 = neural_analysis_helpers.get_rewarded_landmarks(VR_data, nidaq_data, session5)

    # # 5. Find landmark entries by catetory 
    # rew_lm_entry_idx, miss_lm_entry_idx, nongoal_lm_entry_idx, test_lm_entry_idx = \
    #     neural_analysis_helpers.get_landmark_category_entries(VR_data, nidaq_data, sequence, session5)

    # # 6. Find indices of rewards or 'imaginary' rewards in rewarded and non-rewarded landmarks
    # session5 = neural_analysis_helpers.get_landmark_category_rew_idx(sequence, session5, VR_data, nidaq_data)

# Get lick profile 
neural_analysis_helpers.plot_lick_maps(session5)

# Get speed profile
parse_session_functions.plot_speed_profile(session5, stage=5)

# Get speed and lick rate as fake neurons and plot goal progress
lick_neuron = np.array(session5['frame_lick_rate']).reshape(1, -1)
speed_neuron = np.array(session5['speed']).reshape(1, -1)

event_idx = np.sort(np.concatenate([session5['reward_idx'], session5['miss_rew_idx'], session5['test_rew_idx']])).astype(int)

_ = cellTV.plot_arb_progress_2cells(dF=[lick_neuron, speed_neuron], cell=[0, 0], 
                                event_frames=[event_idx, event_idx], ngoals=5, 
                                bins=90, stages=np.array([5, 5]), labels=['T5 - lick rate', 'T5 - speed'], 
                                plot=True, shuffle=False)

In [ ]:
# Goal progress - 4 goals

# Get the neurons in this session that have been tracked with roicat
session_idx = [s for s in range(len(idx_original_aligned)) if str(s) in stage][0] # session idx in the aligned neuron array
tracked_neurons[session_idx] = idx_original_aligned[session_idx][~np.isnan(idx_original_aligned[session_idx])].astype(int)

# Find goal progress tuned cells 
goal_progress_tuned_t5_4g = get_goal_progress_cells(dF_5, tracked_neurons[session_idx], session5, \
                                              event_frames=session5['reward_idx'], stage=session_idx, \
                                                save_path=save_path5, ngoals=4, bins=90, plot=True, \
                                                    shuffle=True, reload=reload_goal_progress)

In [ ]:
# Goal progress - 5 goals

# Get the neurons in this session that have been tracked with roicat
session_idx = [s for s in range(len(idx_original_aligned)) if str(s) in stage][0] # session idx in the aligned neuron array
tracked_neurons[session_idx] = idx_original_aligned[session_idx][~np.isnan(idx_original_aligned[session_idx])].astype(int)

# Find goal progress tuned cells 
event_idx = np.sort(np.concatenate([session5['reward_idx'], session5['miss_rew_idx'], session5['test_rew_idx']])).astype(int)

goal_progress_tuned_t5_5g = get_goal_progress_cells(dF_5, tracked_neurons[session_idx], session5, \
                                              event_frames=event_idx, stage=session_idx, \
                                                save_path=save_path5, ngoals=5, bins=90, plot=True, \
                                                    shuffle=True, reload=reload_goal_progress)

##### Suppression signal analysis

In [ ]:
stage = 5
bins = 30
nreps = 100
shuffle = False
plot_mean_vs_shuffled = False
reload_goal_tuned = False 
reload_goal_lmEntry_tuned = False
goal_by_lmEntry = False

# Select data
session = session5
dF = dF_5
session_idx = stage # session idx in the aligned neuron array
save_path = save_path5

# Define saving paths
test_lm_path = Path(save_path) / 'test_lm_cells' 
test_lm_path.mkdir(parents=True, exist_ok=True)

goal_tuned_path = os.path.join(test_lm_path, f"t{session_idx}_test_goal_tuned.npz")
lm_firing_path = os.path.join(test_lm_path, f't{session_idx}_lm_firing.npz')
goal_lmEntry_firing_path = os.path.join(test_lm_path, f't{session_idx}_goal_lmExit_firing.npz')
goal_firing_path = os.path.join(test_lm_path, f't{session_idx}_goal_firing.npz')
before_lm_firing_path = os.path.join(test_lm_path, f"t{session_idx}_before_lm_firing.npz")

# Load cell IDs if already calculated 
if os.path.exists(goal_tuned_path) and not reload_goal_tuned:
    print(f'Found test-tuned cells for T{session_idx}. Loading ...')
    data = np.load(goal_tuned_path, allow_pickle=True)
    test_peak_cells_t5 = data["high_test_goal_cells"]
else:
    print(f'Did not find test-tuned cells for T{session_idx}.')
    reload_goal_tuned = True

In [ ]:
# Bin neural activity into 5 goals using landmark occupancy indices
# goal_lms = [1,3,5,7,9]
goal_lms = [0,2,4,6,8]

ngoals = len(goal_lms)
bins = 90
nbins = ngoals * bins

# Choose binning data and paths
if goal_by_lmEntry: 
    # Non-goal landmark entry indices are used to bin activity so that 
    print('Using landmark entry events for goal binning')

    lm_entry_idx, lm_exit_idx = neural_analysis_helpers.get_lm_entry_exit(session)
    # event_idx = np.sort(np.concatenate([lm_exit_idx[i::session['num_landmarks']] for i in goal_lms]))
    event_idx = np.sort(np.concatenate([lm_entry_idx[i::session['num_landmarks']] for i in goal_lms]))

    binned_firing_path = goal_lmEntry_firing_path
    reload_flag = reload_goal_lmEntry_tuned
    
else:
    # Reward indices are used to bin activity
    print('Using reward events for goal binning')
    
    event_idx = np.sort(np.concatenate([session['reward_idx'], session['miss_rew_idx'], session['test_rew_idx']])).astype(int)

    binned_firing_path = goal_firing_path
    reload_flag = reload_goal_tuned

# Bin data
nlaps = len(event_idx) // ngoals

tracked_neurons[session_idx] = idx_original_aligned[session_idx][~np.isnan(idx_original_aligned[session_idx])].astype(int)

if os.path.exists(binned_firing_path) and not reload_flag:
    print(f'Found goal activity for T{session_idx}. Loading ...')
    data = np.load(binned_firing_path, allow_pickle=True)
    mean_goal_firing = data["mean_goal_firing"].item()
    shuffled_goal_firing = data["shuffled_goal_firing"].item()

else:
    mean_goal_firing = {}
    shuffled_goal_firing = {}

    for cell in tracked_neurons[session_idx]: 
        if shuffle:
            mean_goal_firing[cell] = cellTV.extract_arb_progress(dF, cell, event_idx, ngoals=ngoals, bins=bins, stage=stage, plot=False, shuffle=False)
            
            shuffled_goal_firing[cell] = np.empty((nreps, nlaps, nbins))
            for i in range(nreps):
                shuffled_goal_firing[cell][i] = cellTV.extract_arb_progress(dF, cell, event_idx, ngoals=ngoals, bins=bins, stage=stage, plot=False, shuffle=shuffle)
        else:
            mean_goal_firing[cell] = cellTV.extract_arb_progress(dF, cell, event_idx, ngoals=ngoals, bins=bins, stage=stage, plot=False, shuffle=False)

    # Save data 
    np.savez(binned_firing_path, 
            mean_goal_firing=mean_goal_firing, 
            shuffled_goal_firing=shuffled_goal_firing)
    
mean_goal_firing5 = mean_goal_firing

In [ ]:
# Find cells with preference for the test landmark 
high_test_tuned_path = os.path.join(test_lm_path, f"t{session_idx}_high_test_goal_tuned.npz")

neurons = goal_progress_tuned_t5_5g
high_test_peak_cells_t5 = get_high_peak_tuned_cells(dF, mean_goal_firing, event_idx, 
                                                    session_idx, neurons, 
                                                    bins=90, rew_goals=[0,1,2,4], 
                                                    test_goal=3, session=session,
                                                    save_path=goal_tuned_path, plot=True, 
                                                    add_lick_rate=True)


In [ ]:
# # GLM to find neurons with higher firing around landmark 10 
# if not reload_test_tuned:
#     if os.path.exists(lm_firing_path):
#         print(f'Found landmark activity for T{session_idx}. Loading ...')
#         data = np.load(lm_firing_path, allow_pickle=True)
#         mean_lm_firing = data["mean_lm_firing"].item()
#         shuffled_lm_firing = data["shuffled_lm_firing"].item()

#     else:
#         lm_entry_idx, lm_exit_idx = neural_analysis_helpers.get_lm_entry_exit(session)

#         tracked_neurons[session_idx] = idx_original_aligned[session_idx][~np.isnan(idx_original_aligned[session_idx])].astype(int)

#         mean_lm_firing = {}
#         shuffled_lm_firing = {}

#         for cell in tracked_neurons[session_idx]: 
            
#             if shuffle:
#                 _, _, mean_lm_firing[cell], _ = neural_analysis_helpers.get_lm_firing(dF, cell, session, lm_entry_idx=lm_entry_idx, lm_exit_idx=lm_exit_idx, bins=bins, shuffle=False)

#                 shuffled_lm_firing[cell] = np.zeros((nreps, session['num_landmarks']))
#                 for i in range(nreps):
#                         _, _, shuffled_lm_firing[cell][i], _ = neural_analysis_helpers.get_lm_firing(dF, cell, session, lm_entry_idx=lm_entry_idx, lm_exit_idx=lm_exit_idx, bins=bins, shuffle=True)
#             else:
#                 _, _, mean_lm_firing[cell], _ = neural_analysis_helpers.get_lm_firing(dF, cell, session, lm_entry_idx=lm_entry_idx, lm_exit_idx=lm_exit_idx, bins=bins, shuffle=False)

#             if plot_mean_vs_shuffled:
#                 fig, ax = plt.subplots(1, session['num_landmarks'], figsize=(12, 3), sharey=True)
#                 ax = ax.ravel()
#                 for i in range(session['num_landmarks']):
#                     ax[i].hist(shuffled_lm_firing[cell][:,i], bins=30, alpha=0.7, color='blue')
#                     ax[i].axvline(mean_lm_firing[cell][i], color='red', linestyle='dashed', linewidth=2, label='Observed Tuning Score')
#                 fig.supxlabel("Tuning Score")
#                 fig.supylabel("Frequency")
#                 plt.suptitle(f'Histogram of Shuffled Tuning Scores for Cell {cell}')
#                 plt.tight_layout()
#                 plt.show()   

#         # Save data 
#         np.savez(lm_firing_path, 
#                 mean_lm_firing=mean_lm_firing, 
#                 shuffled_lm_firing=shuffled_lm_firing)


In [ ]:
# # Find cells with higher activity than shuffle for each landmark
# lm_tunings = [[] for _ in range(session['num_landmarks'])]
# for i in range(session['num_landmarks']):
#     for cell in tracked_neurons[session_idx]: 
#         if mean_lm_firing[cell][i] - np.median(shuffled_lm_firing[cell][i]) > 0.1:
#             lm_tunings[i].append(cell)

# # TODO Find overlap with other tuning method


In [ ]:
# # Find cells with higher test vs rew firing 
# figpath = Path(test_lm_path) / 'lm10_cells'
# figpath.mkdir(parents=True, exist_ok=True)

# high_lm10_cells = neural_analysis_helpers.get_high_lm_firing_cells(dF, lm_tunings[9], session, 
#                                                                    goal_lms=[1,3,5,7], test_lm=9, 
#                                                                    bins=bins, stage=stage, plot=True,
#                                                                    saveplot=True, figpath=figpath)

In [ ]:
# # Find cells with higher lm9 vs lm1357 firing 
# figpath = Path(test_lm_path) / 'lm9_cells'
# figpath.mkdir(parents=True, exist_ok=True)

# high_lm9_cells = neural_analysis_helpers.get_high_lm_firing_cells(dF, lm_tunings[8], session, 
#                                                                   goal_lms=[0,2,4,6], test_lm=8, 
#                                                                   bins=bins, stage=stage, plot=True,
#                                                                   saveplot=True, figpath=figpath)

In [ ]:
# # Find cells with higher before-test vs before-rew firing 
# if not reload_test_tuned:
#     if os.path.exists(before_lm_firing_path):
#         print(f'Found before-landmark activity for T{session_idx}. Loading ...')
#         data = np.load(before_lm_firing_path, allow_pickle=True)
#         mean_lm_firing = data["mean_lm_firing"].item()
#         shuffled_lm_firing = data["shuffled_lm_firing"].item()

#     else:
#         # Get entry and exit indices for the between-landmark points
#         before_lm_entry_idx, before_lm_exit_idx = neural_analysis_helpers.get_before_lm_entry_exit(session)

#         # Find cells with higher than shuffle firing before lm 10
#         mean_lm_firing = {}
#         shuffled_lm_firing = {}

#         for cell in tracked_neurons[session_idx]:     
#             if shuffle:
#                 _, _, mean_lm_firing[cell], _ = neural_analysis_helpers.get_lm_firing(dF, cell, session, lm_entry_idx=before_lm_entry_idx, lm_exit_idx=before_lm_exit_idx, bins=bins, shuffle=False)

#                 shuffled_lm_firing[cell] = np.zeros((nreps, session['num_landmarks']))
#                 for i in range(nreps):
#                         _, _, shuffled_lm_firing[cell][i], _ = neural_analysis_helpers.get_lm_firing(dF, cell, session, lm_entry_idx=before_lm_entry_idx, lm_exit_idx=before_lm_exit_idx, bins=bins, shuffle=True)
#             else:
#                 _, _, mean_lm_firing[cell], _ = neural_analysis_helpers.get_lm_firing(dF, cell, session, lm_entry_idx=before_lm_entry_idx, lm_exit_idx=before_lm_exit_idx, bins=bins, shuffle=False)

#             if plot_mean_vs_shuffled:
#                 fig, ax = plt.subplots(1, session['num_landmarks'], figsize=(12, 3), sharey=True)
#                 ax = ax.ravel()
#                 for i in range(session['num_landmarks']):
#                     ax[i].hist(shuffled_lm_firing[cell][:,i], bins=30, alpha=0.7, color='blue')
#                     ax[i].axvline(mean_lm_firing[cell][i], color='red', linestyle='dashed', linewidth=2, label='Observed Tuning Score')
#                 fig.supxlabel("Tuning Score")
#                 fig.supylabel("Frequency")
#                 plt.suptitle(f'Histogram of Shuffled Tuning Scores for Cell {cell}')
#                 plt.tight_layout()
#                 plt.show() 

#         # Save data 
#         np.savez(before_lm_firing_path, 
#                 mean_lm_firing=mean_lm_firing, 
#                 shuffled_lm_firing=shuffled_lm_firing)


In [ ]:
# # Find cells with higher activity than shuffle before each landmark
# before_lm_tunings = [[] for _ in range(session['num_landmarks'])]
# for i in range(session['num_landmarks']):
#     for cell in tracked_neurons[session_idx]: 
#         if mean_lm_firing[cell][i] - np.median(shuffled_lm_firing[cell][i]) > 0.1:
#             before_lm_tunings[i].append(cell)

# figpath = Path(test_lm_path) / 'before_lm10_cells'
# figpath.mkdir(parents=True, exist_ok=True)

# high_before_lm10_cells = neural_analysis_helpers.get_high_lm_firing_cells(dF, before_lm_tunings[9], session, 
#                                                                    goal_lms=[1,3,5,7], test_lm=9, 
#                                                                    bins=bins, stage=stage,
#                                                                    lm_entry_idx=before_lm_entry_idx, 
#                                                                    lm_exit_idx=before_lm_exit_idx,
#                                                                    plot=True, saveplot=True, figpath=figpath)


In [ ]:
# # Save the cells
# np.savez(test_tuned_path, 
#          high_lm10_cells=high_lm10_cells, 
#          high_lm9_cells=high_lm9_cells,
#          high_before_lm10_cells=high_before_lm10_cells)


What are T6 test-tuned neurons doing in T5? 

In [ ]:
# What are T6 test-tuned neurons doing in T5? 

# Define fig save directory 
save_path = Path(os.path.join(base_path, mouse)) / 't5_t6' / 'goal_progress' / 'test_tuned' / 'figures'
save_path.mkdir(exist_ok=True)

# Find matching cell in T5
test_peak_cells_t6 = np.array(test_peak_cells_t6).astype(int)

t6_neurons_idx = np.where(np.isin(idx_original_aligned[6], test_peak_cells_t6))[0]
valid_neuron_mask = ~np.isnan(idx_original_aligned[5][t6_neurons_idx])
valid_neuron_idx = t6_neurons_idx[valid_neuron_mask]

matching_high_lm10_t5_neurons = idx_original_aligned[5][valid_neuron_idx].astype(int)
matching_high_lm10_t6_neurons = test_peak_cells_t6[valid_neuron_mask]

# Get relevant event data
lm_entry_idx, lm_exit_idx = neural_analysis_helpers.get_lm_entry_exit(session5)

event_idx5 = np.sort(np.concatenate([session5['reward_idx'], session5['miss_rew_idx'], session5['test_rew_idx']])).astype(int)
event_idx6 = np.sort(np.concatenate([session6['reward_idx'], session6['miss_rew_idx'], session6['test_rew_idx']])).astype(int)

# Split T5 into 3 behavioural stages
split_event_idx = {}
for i in range(3):
    if i == 0:
        trials = np.arange(0, 70)   # laps 0-7
    elif i == 1:
        trials = np.arange(70, 200)     # laps 7-20
    elif i == 2:
        trials = np.arange(200, len(session5['all_lms']))

    # Find event indices for the selected trials 
    split_event_idx[i] = [idx for idx in event_idx5 
                          if ((idx >= lm_entry_idx[trials[0]]) & (idx <= lm_entry_idx[trials[-1]]))]

# Plot T5 and T6 activity together
for c, cell5, cell6 in zip(valid_neuron_idx, matching_high_lm10_t5_neurons, matching_high_lm10_t6_neurons):
    # 1. T5 and T6
    fig, axs = plt.subplots(1, 2, subplot_kw={'projection':'polar'}, figsize=(6, 3), facecolor='none')
    neural_analysis_helpers.plot_arb_progress(dF_5, cell5, event_frames=event_idx5, ngoals=5, bins=90, 
                                              stage=5, labels=f'T5 cell {c}', ax=axs[0])

    neural_analysis_helpers.plot_arb_progress(dF_6, cell6, event_frames=event_idx6, ngoals=5, bins=90, 
                                              stage=6, labels=f'T6 cell {c}', ax=axs[-1])
    
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, f'T5_T6_cell{c}_{cell5}_{cell6}.png'), transparent=True, dpi=300)
    
    # 2. Split T5 and T6
    fig, axs = plt.subplots(1, 4, subplot_kw={'projection':'polar'}, figsize=(10, 3), facecolor='none')

    for i in range(3):
        neural_analysis_helpers.plot_arb_progress(dF_5, cell5, event_frames=split_event_idx[i], ngoals=5, 
                                                  bins=90, stage=5, labels=f'T5 stage {i+1} cell {c}', ax=axs[i])

    neural_analysis_helpers.plot_arb_progress(dF_6, cell6, event_frames=event_idx6, ngoals=5, bins=90, 
                                              stage=6, labels=f'T6 cell {c}', ax=axs[-1])
    
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, f'T5split_T6_cell{c}_{cell5}_{cell6}.png'), transparent=True, dpi=300)


In [ ]:
# #  left off here 

# goal_lms = [1,3,5,7]
# test_lm = 9
# stage = 5
# mean_goal_lm_firing = {}
# mean_test_lm_firing = {}
# mean_firing = {}
# binned_firing = {}
# binned_lm_firing = {}
# wilcoxon_stat = np.zeros((len(matching_high_lm10_t5_neurons)))
# wilcoxon_pval = np.zeros((len(matching_high_lm10_t5_neurons)))

# for c, cell in enumerate(matching_high_lm10_t5_neurons):

#     mean_firing[cell], binned_firing[cell], _, binned_lm_firing[cell] = neural_analysis_helpers.get_lm_firing(dF_5, cell, session5, bins=40, shuffle=False)

#     # Gather mean firing for all goal landmarks
#     arrs = [mean_firing[cell][i::session['num_landmarks']] for i in goal_lms]
#     stacked = np.stack(arrs, axis=1)   # shape (n_rows, n_goal_lms)
#     mean_goal_lm_firing[cell] = stacked.mean(axis=1)

#     # Gather mean firing for all goal landmarks
#     mean_test_lm_firing[cell] = np.concatenate([mean_firing[cell][test_lm::session['num_landmarks']]])

#     # Perform Wilcoxon test on test vs goal
#     wilcoxon_stat[c], wilcoxon_pval[c] = stats.wilcoxon(mean_goal_lm_firing[cell], mean_test_lm_firing[cell]) 

# # Find cells with higher test vs goal firing and plot binned firing rate per landmark per lap and polar plot of fiirng rate per lm. 
# high_lm_cells = []
# for c, cell in enumerate(matching_high_lm10_t5_neurons):
#     # Criterion 1: p-value < 0.05
#     if wilcoxon_pval[c] < 0.05:

#         # Criterion 2: max test firing > max goal firing
#         goal_arr = [binned_lm_firing[cell][i::session['num_landmarks'], :] for i in goal_lms]
#         goal_arr = np.stack(goal_arr, axis=1).flatten()
#         test_arr = binned_lm_firing[cell][test_lm::session['num_landmarks'], :].flatten()

#         if np.max(test_arr) > np.max(goal_arr):
            
#             high_lm_cells.append(cell)

#             # if plot:
#     # Format data for plotting
#     arrs = [binned_firing[cell][i::session['num_landmarks'], :] for i in goal_lms]
#     binned_goal_firing = np.stack(arrs, axis=1)   # shape (n_rows, n_goal_lms, n_cols)
#     binned_test_firing = binned_firing[cell][test_lm::session['num_landmarks'], :]
    
#     combined_rows = np.concatenate([binned_goal_firing, binned_test_firing[:, None, :]], axis=1)  # axis1 = landmarks
#     plot_data = combined_rows.reshape(binned_goal_firing.shape[0], -1)

#     # Plot
#     fig = plt.figure(figsize=(10, 4))
#     ax0 = fig.add_subplot(121)
    
#     im = ax0.imshow(plot_data, aspect='auto', cmap='viridis')
#     fig.colorbar(im, ax=ax0, label='Firing rate (Hz)')

#     cols_per_lm = binned_firing[cell].shape[1]  # number of bins per landmark
#     for i in range(1, len(goal_lms) + 1):
#         ax0.axvline(i*cols_per_lm, color='white', linestyle='--', lw=0.5)  # reward LMs

#     tick_positions = np.arange(len(goal_lms) + 1) * cols_per_lm + cols_per_lm/2
#     if test_lm == 9:
#         tick_labels = ['A', 'B', 'C', 'D', 'Test']
#     elif test_lm == 8:
#         tick_labels = ['1', '3', '5', '7', '9']
#     ax0.set_xticks(tick_positions, tick_labels, fontsize=10)
#     ax0.set_xlabel('Landmarks')
#     ax0.set_yticks([0, binned_goal_firing.shape[0]-1])
#     ax0.set_ylabel('Lap')
#     ax0.set_title(f'Cell {cell}, p-value {np.round(wilcoxon_pval[c], 2)}')

#     # Polar subplot
#     data = binned_lm_firing[cell].flatten()
#     if stage == 5:
#         color = 'blue'
#     elif stage == 6:
#         color = 'orange'
#     elif stage == 8:
#         color = 'red'
#     else:
#         color = 'blue'
#     ax1 = fig.add_subplot(122, projection='polar')
#     ax1.set_theta_zero_location('N')
#     ax1.set_theta_direction(-1)
#     angles = np.linspace(0, 2 * np.pi, binned_lm_firing[cell].shape[1] * session['num_landmarks'], endpoint=False)
#     # add the first angle to close the circle
#     angles = np.concatenate((angles, [angles[0]]))
#     avg_bin = np.concatenate((data, [data[0]]))
#     # sem_bin = np.concatenate((sem_bin, [sem_bin[0]]))
#     ax1.plot(angles, avg_bin, color=color, linewidth=2)
#     # ax1.fill_between(angles, avg_bin - sem_bin, avg_bin + sem_bin, color='blue', alpha=0.2)
#     #label the cardinal directions
#     ax1.set_xticks(np.linspace(0, 2 * np.pi, session['num_landmarks'], endpoint=False))
#     ax1.set_xticklabels(np.arange(1,11))
#     ax1.set_title(f'Cell {cell} - Average Firing Rate (Polar)')

#                 # if saveplot is True:
#                 #     plt.savefig(figpath / f'cell{cell}.png')
#                 # plt.show()
    

#### Overlapping goal-progress neurons across sessions


##### T5 vs T8

In [ ]:
# Find indices of neurons in tracked and aligned array
t5_neurons_idx = np.where(np.isin(idx_original_aligned[5], goal_progress_tuned_t5_4g))[0]
t8_neurons_idx = np.where(np.isin(idx_original_aligned[8], goal_progress_tuned_t8))[0]

# Common indices
t5_t8_neurons_idx = np.intersect1d(t5_neurons_idx, t8_neurons_idx)

# Common neurons in each session
t5_neurons = idx_original_aligned[5][t5_t8_neurons_idx]
t8_neurons = idx_original_aligned[8][t5_t8_neurons_idx]

# Save data
tracked_neuron_ids = [t5_neurons, t8_neurons]
tracked_neuron_ids_path_t5_t8_4goal = os.path.join(roicat_dir, 'T5-T8_4goal_progress_tracked_neurons.npz')
np.savez(tracked_neuron_ids_path_t5_t8_4goal, t5_neurons=t5_neurons, t8_neurons=t8_neurons)

In [ ]:
# Plot t5 and t8 peaks together
for c, cell in enumerate(t5_t8_neurons_idx):
    _ = cellTV.plot_arb_progress_2cells(dF=[dF_5, dF_8], cell=[int(t5_neurons[c]), int(t8_neurons[c])], \
                                 event_frames=[session5['reward_idx'], session8['reward_idx']], ngoals=4, 
                                 bins=90, stages=[5, 8], plot=True, shuffle=False)

In [ ]:
# Visualize neurons tracked across t5 and t8

## 4-goal 
# Paths
roicat_dir = roicat_dir
roicat_data_name = str(mouse)
sessions = [sessions_to_align[5],sessions_to_align[8]]
session_keys = ['t5_neurons', 't8_neurons']
tracked_neuron_ids_path = tracked_neuron_ids_path_t5_t8_4goal
dir_save = roicat_dir
filename = str("tracked_t5-t8_4-goal_FOV_clusters")

# Run the script as a subprocess using another conda env
result = subprocess.run(
    [
        roicat_env_python, "roicat_analysis/visualize_selected_tracked_clusters.py",
        roicat_dir,
        roicat_data_name,
        ",".join(sessions),
        tracked_neuron_ids_path,
        ",".join(session_keys),
        dir_save,
        filename
    ],
    capture_output=True, text=True
)

print(result.stdout)
print(result.stderr)

##### T5 vs T6

In [ ]:
# Find indices of neurons in tracked and aligned array
t5_neurons_idx = np.where(np.isin(idx_original_aligned[5], goal_progress_tuned_t5_5g))[0]
t6_neurons_idx = np.where(np.isin(idx_original_aligned[6], goal_progress_tuned_t6_5g))[0]

# Common indices
t5_t6_neurons_idx = np.intersect1d(t5_neurons_idx, t6_neurons_idx)

# Common neurons in each session
t5_neurons = idx_original_aligned[5][t5_t6_neurons_idx]
t6_neurons = idx_original_aligned[6][t5_t6_neurons_idx]

# Save data
tracked_neuron_ids = [t5_neurons, t6_neurons]
tracked_neuron_ids_path_t5_t6_5goal = os.path.join(roicat_dir, 'T5-T6_5goal_progress_tracked_neurons.npz')
# np.savez(tracked_neuron_ids_path_t5_t6_5goal, t5_neurons=t5_neurons, t6_neurons=t6_neurons)

In [ ]:
# Plot t5 and t6 peaks together
event_idx5 = np.sort(np.concatenate([session5['reward_idx'], session5['miss_rew_idx'], session5['test_rew_idx']])).astype(int)
event_idx6 = np.sort(np.concatenate([session6['reward_idx'], session6['miss_rew_idx'], session6['test_rew_idx']])).astype(int)

for c, cell in enumerate(t5_t6_neurons_idx):
    _ = cellTV.plot_arb_progress_2cells(dF=[dF_5, dF_6], cell=[int(t5_neurons[c]), int(t6_neurons[c])], \
                                 event_frames=[event_idx5, event_idx6], ngoals=5, bins=90, stages=[5, 6], plot=True, shuffle=False)

In [ ]:
# Visualize neurons tracked across t5 and t6

## 4-goal 
# Paths
roicat_dir = roicat_dir
roicat_data_name = str(mouse)
sessions = [sessions_to_align[5],sessions_to_align[6]]
session_keys = ['t5_neurons', 't6_neurons']
tracked_neuron_ids_path = tracked_neuron_ids_path_t5_t6_5goal
dir_save = roicat_dir
filename = str("tracked_t5-t6_5-goal_FOV_clusters")

# Run the script as a subprocess using another conda env
result = subprocess.run(
    [
        roicat_env_python, "roicat_analysis/visualize_selected_tracked_clusters.py",
        roicat_dir,
        roicat_data_name,
        ",".join(sessions),
        tracked_neuron_ids_path,
        ",".join(session_keys),
        dir_save,
        filename
    ],
    capture_output=True, text=True
)

print(result.stdout)
print(result.stderr)


##### T5 vs T6 vs T8

In [ ]:
t5_neurons_idx = np.where(np.isin(idx_original_aligned[5], goal_progress_tuned_t5_5g))[0]
t6_neurons_idx = np.where(np.isin(idx_original_aligned[6], goal_progress_tuned_t6))[0]
t8_neurons_idx = np.where(np.isin(idx_original_aligned[8], goal_progress_tuned_t8))[0]

# Common indices
t5_t6_t8_neurons_idx = np.array(list(set(t5_neurons_idx) & set(t6_neurons_idx) & set(t8_neurons_idx)))

# Common neurons in each session
t5_neurons = idx_original_aligned[5][t5_t6_t8_neurons_idx]
t6_neurons = idx_original_aligned[6][t5_t6_t8_neurons_idx]
t8_neurons = idx_original_aligned[8][t5_t6_t8_neurons_idx]

# Save data
tracked_neuron_ids = [t5_neurons, t6_neurons, t8_neurons]
tracked_neuron_ids_path_t5_t6_t8_5goal = os.path.join(roicat_dir, 'T5-T6-T8_5-goal_progress_tracked_neurons.npz')
np.savez(tracked_neuron_ids_path_t5_t6_t8_5goal, t5_neurons=t5_neurons, t6_neurons=t6_neurons, t8_neurons=t8_neurons)

In [ ]:
# Sub-select cells
# T5
event_idx = np.sort(np.concatenate([session5['reward_idx'], session5['miss_rew_idx'], session5['test_rew_idx']])).astype(int)
t5_g5_plotted = []
for c, cell in enumerate(t5_neurons):
    real_score, shuffled_scores, phase_pref, state_pref = cellTV.calc_goal_tuningix(dF_5, int(cell), session5, condition='goal', event_frames=event_idx, n_goals=5, frame_rate=45, bins=90, shuffle=True, plot=False)
    
    if np.abs(real_score - np.median(shuffled_scores)) > 0.5:
        t5_g5_plotted.append(c)
        _ = cellTV.extract_arb_progress(dF_5, int(cell), event_frames=event_idx, ngoals=5, bins=90, plot=False, shuffle=False)

# T6
t6_plotted = []
for c, cell in enumerate(t6_neurons):
    real_score, shuffled_scores, phase_pref, state_pref = cellTV.calc_goal_tuningix(dF_6, int(cell), session6, condition='goal', event_frames=session6['reward_idx'], n_goals=4, frame_rate=45, bins=90, shuffle=True, plot=False)

    if np.abs(real_score - np.median(shuffled_scores)) > 0.5:
        t6_plotted.append(c)
        _ = cellTV.extract_arb_progress(dF_6, int(cell), event_frames=session6['reward_idx'], ngoals=4, bins=90, plot=False, shuffle=False)

# T8
t8_plotted = []
for c, cell in enumerate(t8_neurons):
    real_score, shuffled_scores, phase_pref, state_pref = cellTV.calc_goal_tuningix(dF_8, int(cell), session8, condition='goal', event_frames=session8['reward_idx'], n_goals=4, frame_rate=45, bins=90, shuffle=True, plot=False)

    if np.abs(real_score - np.median(shuffled_scores)) > 0.5:
        t8_plotted.append(c)
        _ = cellTV.extract_arb_progress(dF_8, int(cell), event_frames=session8['reward_idx'], ngoals=4, bins=90, plot=False, shuffle=False)


In [ ]:
# Find common cells that fulfil the criteria 
event_idx = np.sort(np.concatenate([session5['reward_idx'], session5['miss_rew_idx'], session5['test_rew_idx']])).astype(int)

common_idx = np.array(list(set(t5_g5_plotted) & set(t6_plotted) & set(t8_plotted)))

for c in common_idx:
    _ = cellTV.extract_arb_progress(dF_5, int(t5_neurons[c]), event_frames=event_idx, ngoals=5, bins=90, plot=True, shuffle=False)
    _ = cellTV.extract_arb_progress(dF_6, int(t6_neurons[c]), event_frames=session6['reward_idx'], ngoals=4, bins=90, plot=True, shuffle=False)
    _ = cellTV.extract_arb_progress(dF_8, int(t8_neurons[c]), event_frames=session8['reward_idx'], ngoals=4, bins=90, plot=True, shuffle=False)


In [ ]:
# Visualize neurons tracked across t5, t6 and t8
# Paths
roicat_dir = roicat_dir
roicat_data_name = str(mouse)
sessions = [sessions_to_align[5],sessions_to_align[6],sessions_to_align[8]]
session_keys = ['t5_neurons', 't6_neurons', 't8_neurons']
tracked_neuron_ids_path = tracked_neuron_ids_path_t5_t6_t8_5goal
dir_save = roicat_dir
filename = str("tracked_t5-t6-t8_5-goal_FOV_clusters")

# Run the script as a subprocess using another conda env
result = subprocess.run(
    [
        roicat_env_python, "roicat_analysis/visualize_selected_tracked_clusters.py",
        roicat_dir,
        roicat_data_name,
        ",".join(sessions),
        tracked_neuron_ids_path,
        ",".join(session_keys),
        dir_save,
        filename
    ],
    capture_output=True, text=True
)

print(result.stdout)
print(result.stderr)


#### Split T5 into 3 behaviour blocks

In [ ]:
# Get lm entry and exit indices
lm_entry_idx, lm_exit_idx = neural_analysis_helpers.get_lm_entry_exit(session5)

# All event indices
event_idx = np.sort(np.concatenate([session5['reward_idx'], session5['miss_rew_idx'], session5['test_rew_idx']])).astype(int)

# Get tracked neurons
tracked_neurons = idx_original_aligned[5][~np.isnan(idx_original_aligned[5])].astype(int)

# Split trials
split_goal_progress_tuned_t5_5g = {}
split_event_idx = {}
for i in range(3):
    if i == 0:
        trials = np.arange(0, 70)   # laps 0-7
    elif i == 1:
        trials = np.arange(70, 200)     # laps 7-20
    elif i == 2:
        trials = np.arange(200, len(session5['all_lms']))

    # Find event indices for the selected trials 
    split_event_idx[i] = []
    split_event_idx[i].append([idx for idx in event_idx if ((idx >= lm_entry_idx[trials[0]]) & (idx <= lm_entry_idx[trials[-1]]))])

    # Find goal progress tuned cells
    split_goal_progress_tuned_t5_5g[i] = []
    for cell in tracked_neurons:
        if np.isin(cell, tracked_neuron_ids): # make sure it is the correct variable
            real_score, shuffled_scores, phase_pref, state_pref = cellTV.calc_goal_tuningix(dF_5, cell, session5, condition='goal', event_frames=split_event_idx[i], n_goals=5, frame_rate=45, bins=90, shuffle=True, plot=False)

            if np.abs(real_score - np.median(shuffled_scores)) > 0.5:
                split_goal_progress_tuned_t5_5g[i].append(cell)

                # Plot firing rates for goal progress tuned cells
                _ = cellTV.extract_arb_progress(dF_5, cell, event_frames=split_event_idx[i], ngoals=5, bins=90, plot=True, shuffle=False)


In [ ]:
for cell in split_goal_progress_tuned_t5_5g[0]:

    # Plot firing rates for goal progress tuned cells
    _ = cellTV.extract_arb_progress(dF_5, cell, event_frames=split_event_idx[0][0], ngoals=5, bins=90, plot=True, shuffle=False)


In [ ]:
for cell in split_goal_progress_tuned_t5_5g[1]:

    # Plot firing rates for goal progress tuned cells
    _ = cellTV.extract_arb_progress(dF_5, cell, event_frames=split_event_idx[1][0], ngoals=5, bins=90, plot=True, shuffle=False)


In [ ]:
for cell in split_goal_progress_tuned_t5_5g[2]:

    # Plot firing rates for goal progress tuned cells
    _ = cellTV.extract_arb_progress(dF_5, cell, event_frames=split_event_idx[2][0], ngoals=5, bins=90, plot=True, shuffle=False)


#### Correlation analysis without goal progress cells in T5

In [ ]:
# Select neurons
remaining_neurons = np.setdiff1d(np.arange(0, dF_5.shape[0]), goal_progress_tuned_t5_5g)

# Correlation params and data
num_lms_considered = np.round((len(session5['all_lms']) // 10) * 10)
ABCD_goals = [1,2,3,4]
all_reward_idx = np.sort(np.concatenate([session5['reward_idx'], session5['miss_rew_idx'], session5['nongoal_rew_idx'], session5['test_rew_idx']]))

# Get landmark PSTH
landmark_psth, average_landmark_psth = neural_analysis_helpers.get_landmark_psth(data=dF_5, neurons=remaining_neurons, event_idx=all_reward_idx[:num_lms_considered], \
                                                            num_landmarks=session5['num_landmarks'], time_around=1)
        
window_size = 5     # laps 

# Get rolling reward avg
rew_idx = {}    
for g, goal in enumerate(ABCD_goals):
    rew_idx[goal] = np.array([idx for idx in session5['rewarded_landmarks'] \
                            if session5['all_lms'][idx] == session5['goal_landmark_id'][goal-1]])
    
    num_windows = len(rew_idx[goal]) 

rolling_avg_reward_psth = np.zeros((landmark_psth.shape[0], num_windows-window_size, landmark_psth.shape[2])) 

for i in range(num_windows-window_size):
    psths_considered = np.concatenate([landmark_psth[:, rew_idx[goal][i]:rew_idx[goal][i+window_size]:session5['num_landmarks'], :] for goal in ABCD_goals], axis=1)
    # shape: num_neurons x window_size * num_goals x num_timebins

    rolling_avg_reward_psth[:, i, :] = np.mean(psths_considered, axis=1) 
    # shape: num_neurons x num_windows x num_timebins

# Get rolling test avg
valid_test_indices = np.array([idx for idx in session5['test_idx'] if (idx > session5['rewarded_landmarks'][0] and idx < session5['rewarded_landmarks'][-1]+3)])
num_windows = int(len(valid_test_indices - window_size + 1))
rolling_avg_test_psth = np.zeros((landmark_psth.shape[0], num_windows-window_size, landmark_psth.shape[2])) 

for i in range(num_windows-window_size):
    rolling_avg_test_psth[:, i, :] = np.mean(landmark_psth[:, valid_test_indices[i]:valid_test_indices[i+window_size]:session5['num_landmarks'], :], axis=1)

# Get the pairwise correlations of rolling lap blocks
conditions = ['reward', 'test']
average_psths = [rolling_avg_reward_psth, rolling_avg_test_psth]

# Get correlation matrix of average psths in rolling lap blocks
similarity_matrices = neural_analysis_helpers.get_window_similarity_matrix(average_psths, conditions, population=False, zscoring=True, plot=True)

#### Licking analysis

##### Lick rate polar plots

In [ ]:
# Get polar plots for lick rate 
event_idx = np.sort(np.concatenate([session5['reward_idx'], session5['miss_rew_idx'], session5['test_rew_idx']])).astype(int)

dF = np.array(session5['frame_lick_rate']).reshape(1, -1)
real_score, shuffled_scores, phase_pref, state_pref = cellTV.calc_goal_tuningix(dF, 0, session5, condition='goal', event_frames=event_idx, n_goals=5, frame_rate=45, bins=90, shuffle=True, plot=True)

_ = cellTV.extract_arb_progress(dF, 0, event_idx, ngoals=5, bins=90, plot=True, shuffle=False)


##### T5: neural activity around 1st lick in landmark

In [ ]:
# Plot lick-bound neural activity
first_licks, _ = neural_analysis_helpers.get_first_licks(session5)
first_licks = {id: np.array(lst) for id, lst in first_licks.items()}

# Landmark licks and non-licks
lick_ids = [1, 2, 4, 8, 10]
lick_events = np.sort(np.concatenate([first_licks[i] for i in lick_ids]))

# Landmark rewards 
event_idx = np.sort(np.concatenate([session5['reward_idx'], session5['miss_rew_idx'], session5['test_rew_idx']])).astype(int)

# Get the neurons in this session that have been tracked with roicat
stage = 't5'
session_idx = [s for s in range(len(idx_original_aligned)) if str(s) in stage][0] # session idx in the aligned neuron array

tracked_neurons[session_idx] = idx_original_aligned[session_idx][~np.isnan(idx_original_aligned[session_idx])].astype(int)

# Plot reward and lick-bound activity together   
for c, neuron in enumerate(goal_progress_tuned_t5_5g):
    _ = cellTV.plot_arb_progress_2cells(dF=[dF_5, dF_5], cell=[int(neuron), int(neuron)], \
                                 event_frames=[event_idx, lick_events], ngoals=5, 
                                 bins=90, stages=np.array([5, 5]), plot=True, shuffle=False)

In [ ]:
# Plot reward and lick rate together 
dF = np.array(session5['frame_lick_rate']).reshape(1, -1)

for c, neuron in enumerate(goal_progress_tuned_t5_5g):
    _ = cellTV.plot_arb_progress_2cells(dF=[dF_5, dF], cell=[int(neuron), 0], 
                                 event_frames=[event_idx, event_idx], ngoals=5, 
                                 bins=90, stages=np.array([5, 5]), labels=['T5 - rewards', 'T5 - lick rate'], 
                                 plot=True, shuffle=False)

##### T6: neural activity around 1st lick in landmark

In [ ]:
# Plot lick-bound neural activity
first_licks, _ = neural_analysis_helpers.get_first_licks(session6)
first_licks = {id: np.array(lst) for id, lst in first_licks.items()}

# Landmark licks and non-licks
lick_ids = [1, 2, 4, 8, 10]
lick_events = np.sort(np.concatenate([first_licks[i] for i in lick_ids]))

# Landmark rewards
event_idx = np.sort(np.concatenate([session6['reward_idx'], session6['miss_rew_idx'], session6['test_rew_idx']])).astype(int)

# Get the neurons in this session that have been tracked with roicat
stage = 't6'
session_idx = [s for s in range(len(idx_original_aligned)) if str(s) in stage][0] # session idx in the aligned neuron array

tracked_neurons[session_idx] = idx_original_aligned[session_idx][~np.isnan(idx_original_aligned[session_idx])].astype(int)

# Plot reward and lick-bound activity together  
for c, neuron in enumerate(goal_progress_tuned_t6_5g):
  _ = cellTV.plot_arb_progress_2cells(dF=[dF_6, dF_6], cell=[int(neuron), int(neuron)], \
                                  event_frames=[event_idx, lick_events], ngoals=5, 
                                  bins=90, stages=np.array([6, 6]), plot=True, shuffle=False)

In [ ]:
# Visually confirm the lick IDs make sense
import matplotlib.pyplot as plt
import palettes
hfs_palette = np.array(palettes.met_brew('Austria',n=10, brew_type="continuous"))

# Load data 
base_path = parse_session_functions.find_base_path_npz(session6['mouse'], session6['date'])
nidaq_data = parse_session_functions.load_session_npz(base_path)
positions = nidaq_data['position']

base_path2 = parse_session_functions.find_base_path(session6['mouse'], session6['date'])
VR_data = parse_session_functions.load_session(base_path2)

# Find licks 
licks = neural_analysis_helpers.get_lick_types(session6, VR_data, nidaq_data)

# Find landmark entry and exit
lm_entry_idx, lm_exit_idx = neural_analysis_helpers.get_lm_entry_exit(session6, positions=positions)

session6 = neural_analysis_helpers.get_rewarded_landmarks(VR_data, nidaq_data, session6)

# Define example runs 
example_runs = [np.arange(0, lm_exit_idx[9]+1), np.arange(lm_entry_idx[session6['rewarded_landmarks'][10]-1], lm_exit_idx[session6['rewarded_landmarks'][13]+3]+1)]
# example_runs = [np.arange(0, lm_exit_idx[9]+1), np.arange(62000, 67000)]

# Plot
fig, ax = plt.subplots(2, 1, figsize=(12,8), sharey=True)
ax = ax.ravel()

for i in range(2):
    entries = np.array(lm_entry_idx)[(np.array(lm_entry_idx) >= example_runs[i][0]) & (np.array(lm_entry_idx) <= example_runs[i][-1])]
    exits = np.array(lm_exit_idx)[(np.array(lm_exit_idx) >= example_runs[i][0]) & (np.array(lm_exit_idx) <= example_runs[i][-1])]
    
    ax[i].plot(example_runs[i], positions[example_runs[i]], color='gray', alpha=0.5)
    ax[i].scatter(entries, positions[entries], s=200, marker='|', color='k', label='lm entry')
    ax[i].scatter(exits, positions[exits], s=50, marker='|', color='k', label='lm exit')
    
    for id in range(1,11):
        licks_in_range = licks[id][(licks[id] >= example_runs[i][0]) & (licks[id] <= example_runs[i][-1])]
        ax[i].scatter(licks_in_range, positions[licks_in_range], s=10, color=hfs_palette[id-1], label=f'{id}')
    
    ax[i].legend()
    ax[i].set_ylabel('Position')
ax[1].set_xlabel('Time')

# Make sure the missing licks, if any, are only at the very end 
all_licks = np.sort(np.concatenate([licks[id] for id in range(1,8)]))
lick_idx = np.where(nidaq_data['licks'] == 1)[0]

missing_licks = np.setxor1d(lick_idx, all_licks)
fig, ax = plt.subplots(1, 1, figsize=(8,3))
ax.plot(positions, color='gray', alpha=0.5)
ax.scatter(missing_licks, positions[missing_licks], s=5, color='k')

In [ ]:
# Plot reward and lick rate together 
dF = np.array(session6['frame_lick_rate']).reshape(1, -1)

for c, neuron in enumerate(goal_progress_tuned_t6_5g):
    _ = cellTV.plot_arb_progress_2cells(dF=[dF_6, dF], cell=[int(neuron), 0], \
                                 event_frames=[event_idx, event_idx], ngoals=5, 
                                 bins=90, stages=np.array([6, 6]), labels=['T6 - rewards', 'T6 - lick rate'], plot=True, shuffle=False)

### Template correlation analysis
The goal is to identify 4-peak vs 5-peak neurons by correlating them with a 4-peak or a 5-peak template. The neurons are classified based on the correlation with the template the yields the highest correlation. The number of peaks on the smoothed binned firing is added as an additional criterion to correct potential misclassifications. 

In [ ]:
# Create 2 templates - 4 and 5 peaks - of equal size using the lick rate 
templates = []

sessions = [session5, session6]
ngoals = [5, 5]
stages = [5, 6]

for session, goals, stage in zip(sessions, ngoals, stages):
    # Check licking 
    _, _ = neural_analysis_helpers.plot_lick_maps(session)

    # Use lick rate as template
    template = np.array(session['frame_lick_rate']).ravel()

    # Define the template size
    bins = 90
    template_size = 4 * bins
    bin_size = len(template) // template_size   # integer bin size

    # Extract the lick rate binned by goal 
    dF = np.array(session['frame_lick_rate']).reshape(1, -1)
    cell = 0
    event_idx = np.sort(np.concatenate([session['reward_idx'], session['miss_rew_idx'], session['test_rew_idx']])).astype(int)

    binned_lick_rate = cellTV.extract_arb_progress(
        dF, cell, event_idx, ngoals=goals, bins=90, plot=True, shuffle=False)

    avg_binned_lick_rate = np.nanmean(binned_lick_rate, axis=0)
    std_binned_lick_rate = np.nanstd(binned_lick_rate, axis=0)
    sem_binned_lick_rate = std_binned_lick_rate / np.sqrt(binned_lick_rate.shape[0])

    # Downsize to template size if needed
    if avg_binned_lick_rate.shape[0] > template_size:
        template_resized = resample(avg_binned_lick_rate, template_size)
    else:
        template_resized = avg_binned_lick_rate
        
    templates.append(template_resized)

    # Plot the templates 
    N_original = len(avg_binned_lick_rate)
    N_resized = len(template_resized)
    plt.figure(figsize=(8,4))
    if stage == 6:
        colors = ['orange', 'gold']
    elif stage == 5:
        colors = ['blue', 'deepskyblue']
    plt.plot(avg_binned_lick_rate, color=colors[0], label='original lick rate')
    plt.plot(template_resized, color=colors[1], label='resized lick rate')
    plt.xticks([0, N_original, N_resized])
    plt.xlabel('Length of lick rate template')
    plt.ylabel('Lick rate (template)')
    plt.title('Template (lick rate) used to test the goal-progress neural activity')
    plt.legend()


In [ ]:
# Find the cells with highest 4 or 5-fold correlation             
neurons_4peaks_t6, neurons_5peaks_t6 = neural_analysis_helpers.classify_4_or_5_peak_neurons(
                                                                    neurons=goal_progress_tuned_t6_5g, 
                                                                    mean_goal_firing=mean_goal_firing6,
                                                                    peaks=[4,5], plot=False)

neurons_4peaks_t5, neurons_5peaks_t5 = neural_analysis_helpers.classify_4_or_5_peak_neurons(
                                                                    neurons=goal_progress_tuned_t5_5g, 
                                                                    mean_goal_firing=mean_goal_firing5,
                                                                    peaks=[4,5], plot=False)

In [ ]:
# # Get the ACG and CCG with the two templates
# event_idx5 = np.sort(np.concatenate([session5['reward_idx'], session5['miss_rew_idx'], session5['test_rew_idx']])).astype(int)

# acg = {}
# ccg0 = {}
# ccg1 = {}

# for cell in goal_progress_tuned_t5_5g:
#     acg[cell], ccg0[cell], ccg1[cell] = neural_analysis_helpers.get_acg_template_ccg(
#         dF=dF_5, cell=cell, event_idx=event_idx5, ngoals=5, templates=templates, plot_firing=True, plot_corr=True
#     )

# # Find the cells with highest 4 or 5-fold correlation 
# neurons_4peaks_t5 = []
# neurons_5peaks_t5 = []

# for cell in goal_progress_tuned_t5_5g:
#     if np.max(ccg0[cell]) > np.max(ccg1[cell]):
#         neurons_5peaks_t5.append(cell)
#     else:
#         neurons_4peaks_t5.append(cell)

# for cell in neurons_4peaks_t5:
#     _ = cellTV.extract_arb_progress(dF_5, cell, event_idx5, ngoals=5, bins=90, plot=True, shuffle=False)


In [ ]:
# What happens to t5 neurons in t6? 
reload = True

# Load tracked cells
tracked_neuron_ids_path_t5_t6_5goal = os.path.join(roicat_dir, 'T5-T6_5goal_progress_tracked_neurons.npz')
if os.path.exists(tracked_neuron_ids_path_t5_t6_5goal):
    file = np.load(tracked_neuron_ids_path_t5_t6_5goal)
    
    tracked_neurons_t5_g5 = file['t5_neurons']
    tracked_neurons_t6_g5 = file['t6_neurons']
    
# Load categories of cells
save_path = Path(os.path.join(base_path, mouse)) / 't5_t6' / 'goal_progress' 

if os.path.exists(os.path.join(save_path, 'T5_T6_4peak_5peak_neurons.npz')) and not reload:
    data = np.load(os.path.join(save_path, 'T5_T6_4peak_5peak_neurons.npz')) 
    still_four_peaks = data['four_peaks']
    still_five_peaks = data['five_peaks']
    four_to_five_peaks = data['four_to_five_peaks']
    five_to_four_peaks = data['five_to_four_peaks']

else:
        
    # Check if number of peaks changed 
    still_four_peaks = []
    still_five_peaks = []
    four_to_five_peaks = []
    five_to_four_peaks = []

    for c, cell in enumerate(tracked_neurons_t5_g5):
        if np.isin(cell, neurons_4peaks_t5):
            # Check t6 neurons
            if np.isin(tracked_neurons_t6_g5[c], neurons_4peaks_t6):
                still_four_peaks.append((cell, tracked_neurons_t6_g5[c]))
            elif np.isin(tracked_neurons_t6_g5[c], neurons_5peaks_t6):
                four_to_five_peaks.append((cell, tracked_neurons_t6_g5[c]))
        elif np.isin(cell, neurons_5peaks_t5):
            # Check t6 neurons
            if np.isin(tracked_neurons_t6_g5[c], neurons_4peaks_t6):
                five_to_four_peaks.append((cell, tracked_neurons_t6_g5[c]))
            elif np.isin(tracked_neurons_t6_g5[c], neurons_5peaks_t6):
                still_five_peaks.append((cell, tracked_neurons_t6_g5[c]))

    still_four_peaks = np.array(still_four_peaks).astype(int)
    still_five_peaks = np.array(still_five_peaks).astype(int)
    four_to_five_peaks = np.array(four_to_five_peaks).astype(int)
    five_to_four_peaks = np.array(five_to_four_peaks).astype(int)

    # Save the results 
    save_path.mkdir(parents=True, exist_ok=True)

    np.savez(os.path.join(save_path, 'T5_T6_4peak_5peak_neurons.npz'), 
            four_peaks=still_four_peaks,
            five_peaks=still_five_peaks,
            four_to_five_peaks=four_to_five_peaks,
            five_to_four_peaks=five_to_four_peaks)


In [ ]:
# Plot 5 --> 4 peak cells 
for cells in five_to_four_peaks:
    _ = cellTV.plot_arb_progress_2cells(dF=[dF_5, dF_6], cell=cells, \
                                    event_frames=[event_idx5, event_idx6], ngoals=5, bins=90, \
                                    stages=[5, 6], labels=None, plot=True, shuffle=False)

In [ ]:
# Plot 4 --> 4 peak cells 
for cells in still_four_peaks:
    _ = cellTV.plot_arb_progress_2cells(dF=[dF_5, dF_6], cell=cells, \
                                    event_frames=[event_idx5, event_idx6], ngoals=5, bins=90, \
                                    stages=[5, 6], labels=None, plot=True, shuffle=False)

In [ ]:
# Plot 5 --> 5 peak cells 
for cells in still_five_peaks:
    _ = cellTV.plot_arb_progress_2cells(dF=[dF_5, dF_6], cell=cells, \
                                    event_frames=[event_idx5, event_idx6], ngoals=5, bins=90, \
                                    stages=[5, 6], labels=None, plot=True, shuffle=False)

In [ ]:
# Plot 4 --> 5 peak cells 
for cells in four_to_five_peaks:
    _ = cellTV.plot_arb_progress_2cells(dF=[dF_5, dF_6], cell=cells, \
                                    event_frames=[event_idx5, event_idx6], ngoals=5, bins=90, \
                                    stages=[5, 6], labels=None, plot=True, shuffle=False)

In [ ]:
# Find fraction of cells in each category
perc_still_five_peaks = len(still_five_peaks) / len(tracked_neurons_t5_g5)
perc_five_to_four_peaks = len(five_to_four_peaks) / len(tracked_neurons_t5_g5)
print('Still 5 peaks:', perc_still_five_peaks, '\nSwitched from 5 to 4 peaks:', perc_five_to_four_peaks)

##### 1 peak neurons


In [ ]:
# Get the ACG and CCG with the two templates
event_idx6 = np.sort(np.concatenate([session6['reward_idx'], session6['miss_rew_idx'], session6['test_rew_idx']])).astype(int)

acg = {}
ccg0 = {}
ccg1 = {}

for cell in high_lm10_cells:
    acg[cell], ccg0[cell], ccg1[cell] = neural_analysis_helpers.get_acg_template_ccg(
        dF=dF_6, cell=cell, event_idx=event_idx6, ngoals=5, templates=[templates[0], one], plot_firing=False, plot_corr=True
    )

# Find the cells with highest 4 or 5-fold correlation 
lm10_neurons_1peaks_t6 = []
lm10_neurons_5peaks_t6 = []

for cell in high_lm10_cells:
    if np.max(ccg0[cell]) > np.max(ccg1[cell]):
        lm10_neurons_5peaks_t6.append(cell)
    else:
        lm10_neurons_1peaks_t6.append(cell)

for cell in lm10_neurons_1peaks_t6:
    _ = cellTV.extract_arb_progress(
        dF_6, cell, event_idx6, ngoals=5, bins=90, plot=True, shuffle=False)


##### Mock data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy

def circular_crosscorr(x, y):
    return [np.corrcoef(x, np.roll(y, lag))[0,1] for lag in range(len(y))]

def circular_crosscorr2(x, y, normalize=True):
    X = np.fft.fft(x)
    Y = np.fft.fft(y)
    corr = np.fft.ifft(X * Y.conj()).real
    if normalize:
        norm = np.sqrt(np.sum(x**2) * np.sum(y**2))
        corr = corr / norm
    return corr

def circular_crosscorr3(x, y):
    X = np.fft.fft(x - np.mean(x))
    Y = np.fft.fft(y - np.mean(y))
    corr = np.fft.ifft(X * Y.conj()).real
    # shift so that lag=0 is first
    corr = np.fft.fftshift(corr)
    # normalize like Pearson correlation
    corr = corr / (np.std(x) * np.std(y) * len(x))
    return corr


times = np.arange(0, 360)
four = np.zeros((360))
five = np.zeros((360))

# Arbitrarily adds 4 our 5 peaks to simulate activity, scaled to max at 1:
fake_peak = 25*scipy.stats.norm.pdf(range(50), 25, 10)
for i in range(5):
    five[72*i:72*i + 50] = fake_peak
    if i < 4:
        four[72*i:72*i + 50] = fake_peak



plt.plot(times, four)
plt.plot(circular_crosscorr3(four, four))
plt.show()

plt.plot(times, five)
plt.plot(times, circular_crosscorr3(five, five))
plt.show()


# ----------------------- #
# # Cell data 
# # Extract the firing rate
# binned_firing_rate = cellTV.extract_arb_progress(dF_5, 538, event_idx5, ngoals=5, bins=90, plot=False, shuffle=False)
# avg_binned_firing_rate = np.mean(binned_firing_rate, axis=0)

# # Resize firing rate if needed
# template_size = four.shape[0]
# if avg_binned_firing_rate.shape[0] > template_size:
#     firing_rate_resized = resample(avg_binned_firing_rate, template_size)
# else:
#     firing_rate_resized = avg_binned_firing_rate

# # # should be 4 peaks, and an autocorrelogram that peaks at 0 and otherwise has
# # # four other slightly smaller peaks:
# plt.plot(times, four)
# a = circular_crosscorr(firing_rate_resized, four)
# plt.plot(times, a)
# plt.show()

# # should be 5 peaks, and an autocorrelogram that is basically the same but
# # smoother and offset to peak at 0
# plt.plot(times, five)
# b = circular_crosscorr(firing_rate_resized, five)
# plt.plot(times, b)
# plt.show()

# print(np.max(a), np.max(b))
# # # Five identical peaks, since it's "always missing one"!
# # plt.plot(times, circular_crosscorr(four, five))
# # plt.show()

# # # should be 4 peaks, and an autocorrelogram that peaks at 0 and otherwise has
# # # four other slightly smaller peaks:
# plt.plot(times, four)
# c = circular_crosscorr3(firing_rate_resized, four)
# plt.plot(times, c)
# plt.show()

# # should be 5 peaks, and an autocorrelogram that is basically the same but
# # smoother and offset to peak at 0
# plt.plot(times, five)
# d = circular_crosscorr3(firing_rate_resized, five)
# plt.plot(times, d)
# plt.show()

# print(np.max(c), np.max(d))

# # # Five identical peaks, since it's "always missing one"!
# # plt.plot(times, circular_crosscorr2(four, five))
# # plt.show()

### Find 5-peak neurons with strongest test peak

In [ ]:
def get_high_peak_tuned_cells(dF, goal_firing, event_idx, session_idx, neurons, bins,
                         rew_goals, test_goal, session, save_path, plot=True, add_lick_rate=False):
    """
    Find neurons with stronger test-goal than rew-goal tuning using multiple criteria.
    - rew_goals: list of goal indices considered as "reward"
    - test_goal: index of the goal to test against
    """

    high_test_goal_cells = []

    for c, cell in enumerate(neurons):

        # Split data by goal
        goal_data = [
            goal_firing[cell][:, bins*i:bins*(i+1)]
            for i in range(len(rew_goals) + 1)  # total goals
        ]
        
        rew_data = np.hstack([goal_data[i] for i in rew_goals])  # concat reward goals
        test_data = goal_data[test_goal]

        # Mean activity per lap
        mean_rew_data = np.mean(rew_data, axis=1)
        mean_test_data = np.mean(test_data, axis=1)

        # Mean activity per bin
        mean_bin_rew_data = np.mean(rew_data, axis=0)
        mean_bin_test_data = np.mean(test_data, axis=0)

        # Mean activity per bin per goal 
        mean_bin_goal_rew_data = mean_bin_rew_data.reshape(len(rew_goals), bins)

        # Condition 2: test firing stronger than rew
        if np.max(mean_bin_test_data) > 1.2 * np.mean(np.max(mean_bin_goal_rew_data, axis=1)):
            
            high_test_goal_cells.append(cell)

            if plot:
                if add_lick_rate:
                    dF_lick = np.array(session['frame_lick_rate']).reshape(1, -1)

                    cellTV.plot_arb_progress_2cells(dF=[dF, dF_lick], cell=[cell, 0], event_frames=[event_idx, event_idx], 
                                                    ngoals=len(rew_goals)+1, bins=bins, 
                                                    stages=[session_idx, session_idx], labels=[f'cell {cell}', 'lick rate'], 
                                                    plot=True, shuffle=False)
                else:
                    neural_analysis_helpers.plot_arb_progress(dF, cell, event_idx, len(rew_goals) + 1, bins, session_idx, ax=None)

    if save_path is not None:                         
        np.savez(save_path, high_test_goal_cells=high_test_goal_cells)

    return high_test_goal_cells

In [ ]:
# Identify the cells with a stronger test peak in goal progress cells
# high_test_tuned_path = os.path.join(test_lm_path, f"t{session_idx}_high_test_goal_tuned.npz")

# goal_lms = [0,2,4,6,8]
# lm_entry_idx, lm_exit_idx = neural_analysis_helpers.get_lm_entry_exit(session)
# event_idx = np.sort(np.concatenate([lm_entry_idx[i::session['num_landmarks']] for i in goal_lms]))

neurons = goal_progress_tuned_t6_5g
high_test_peak_cells_t6 = get_high_peak_tuned_cells(dF_6, mean_goal_firing6, event_idx6, 
                                                                           session_idx, neurons, 
                                                                           bins=90, rew_goals=[0,1,2,4], 
                                                                           test_goal=3, session=session6,
                                                                           save_path=None, plot=True, 
                                                                           add_lick_rate=True)
# TODO savefig 
# plt.tight_layout()
# plt.savefig(os.path.join(save_path, f'T5split_T6_cell{c}_{cell5}_{cell6}.png'), transparent=True, dpi=300)

# Find fraction of these cells across all session tracked cells 
# perc_test_tuned_t6 = len(high_test_peak_cells_t6) / len(neurons)
# print(perc_test_tuned_t6)